# Repository Evidence Audit

## tl;dr

- The checkout contains **6 static result JSONL files** and **6 rescored JSONL files**, but **0 stored co-evolution result artifacts** and **0 checkpoint or adapter archives**.
- Under the documented Agent-C `deadline` split recreation, **202 of 300** evaluated questions fall in Agent-C training and **38** fall in Agent-C validation. This means the repository does not establish a held-out test set for the static `n=300` evaluation.
- The saved result directory contains **3 exact duplicate file group(s)**. Duplicated filenames must not be treated as independent experimental runs.
- The stored static outputs are suitable for rescoring and descriptive ablations. They are not sufficient on their own to substantiate the submitted 30-iteration, three-seed, seven-benchmark claims.


## Context & Methods

This notebook audits the repository-local evidence intended to support the NeurIPS rebuttal. The unit of analysis is one saved model-response row per question. Checks cover completeness, uniqueness, target variation, exact file duplication, score reproducibility, agent-output coverage, and overlap between Agent-C specialization data and the static evaluation set.

### Key Assumptions

- Agent-C used the documented `deadline` profile: seed 42, at most 2,500 rows, and a 15% validation fraction.
- The `n=300` experiment selected the first 300 rows from `mmlu_pro_hard_subset.json`, matching the repository evaluation loader.
- Repository-local files are the complete evidence available for this audit. External Kaggle artifacts are treated as missing until attached.


In [1]:
from pathlib import Path
import json
import pandas as pd

from audit_core import ARTIFACT_DIR, REPO_ROOT, run_audit

pd.set_option("display.max_columns", 50)
audit = run_audit(write_artifacts=True)
print("Repository:", REPO_ROOT)
print("Artifacts:", ARTIFACT_DIR)


Repository: /Users/ebadshabbir/Desktop/DIVERSITY_NEURIPS/Multi_agent_neurips
Artifacts: /Users/ebadshabbir/Desktop/DIVERSITY_NEURIPS/Multi_agent_neurips/rebuttal/artifacts


## Data

The audit uses the checked-in Experiment 0 JSONL files, rescored JSONLs, MMLU-Pro hard-subset data, and GPQA preparation output. It does not download models or infer missing results.


In [2]:
result_profiles = pd.DataFrame(audit["result_profiles"])
result_profiles[[
    "file", "rows", "unique_question_ids", "duplicate_question_rows",
    "difficulty_complete_rate", "difficulty_unique_values",
    "three_agent_complete_rows", "option_probability_complete_rows",
    "raw_spearman_rho", "revised_spearman_rho"
]]


,file,rows,unique_question_ids,duplicate_question_rows,difficulty_complete_rate,difficulty_unique_values,three_agent_complete_rows,option_probability_complete_rows,raw_spearman_rho,revised_spearman_rho
0,experiments/experiment_0/results/exp0_heteroge...,50,50,0,1.0,35,50,50,0.099618,NaN
1,experiments/experiment_0/results/exp0_heteroge...,300,300,0,1.0,188,300,300,-0.005435,NaN
2,experiments/experiment_0/results/exp0_heteroge...,300,300,0,1.0,188,300,300,-0.005435,NaN
3,experiments/experiment_0/results/exp0_heteroge...,50,50,0,1.0,35,50,50,0.283236,NaN
4,experiments/experiment_0/results/exp0_homogene...,109,109,0,1.0,67,0,0,0.308978,NaN
5,experiments/experiment_0/results/exp0_homogene...,300,300,0,1.0,188,0,0,0.360381,NaN
6,experiments/experiment_0/local_kaggle1_outputs...,300,300,0,1.0,188,300,300,-0.005435,0.002863
7,experiments/experiment_0/local_kaggle1_outputs...,50,50,0,1.0,35,50,50,0.283236,-0.008121
8,experiments/experiment_0/local_kaggle1_outputs...,300,300,0,1.0,188,300,300,-0.005435,0.002863
9,experiments/experiment_0/local_kaggle1_outputs...,50,50,0,1.0,35,50,50,0.099618,0.153920


## Results

### Saved outputs have useful static coverage but do not contain co-evolution trajectories

The table below distinguishes stored static predictions from the iteration-level artifacts and checkpoints required to reproduce Table 3.


In [3]:
inventory = audit["artifact_inventory"]
pd.DataFrame([
    {"artifact": "Static result JSONLs", "count": inventory["static_result_jsonl_count"]},
    {"artifact": "Rescored result JSONLs", "count": inventory["rescored_result_jsonl_count"]},
    {"artifact": "Co-evolution result artifacts", "count": inventory["coevolution_result_artifact_count"]},
    {"artifact": "Checkpoint or adapter archives", "count": inventory["checkpoint_or_adapter_archive_count"]},
])


,artifact,count
0,Static result JSONLs,6
1,Rescored result JSONLs,6
2,Co-evolution result artifacts,0
3,Checkpoint or adapter archives,0


### Some result filenames are exact copies, not independent runs

Exact SHA-256 matches identify copied artifacts. These files can be aliases for convenience, but they must not be counted as separate seeds or replications.


In [4]:
hash_groups = pd.DataFrame(audit["hash_groups"])
hash_groups.loc[hash_groups["exact_duplicate_group"], ["file_count", "files", "sha256"]]


,file_count,files,sha256
0,2,experiments/experiment_0/local_kaggle1_outputs...,4f9e128c670398b0076043d6f1f58db632d4ed51c735aa...
3,2,experiments/experiment_0/local_kaggle1_outputs...,01ba4719c80b6fe911b091a7c05124b64eeece964e09c0...
5,2,experiments/experiment_0/results/exp0_heteroge...,734024ad1cbe515bfec171961a678000f5ca4c7dbc3b95...


### The available Agent-C split overlaps the static evaluation

This reconstruction follows the checked-in Agent-C loader exactly: shuffle the full hard subset with seed 42, retain at most 2,500 questions, allocate the first 15% to validation, and use the rest for training. The experiment loader then evaluates the first 300 hard-subset rows.


In [5]:
overlap = audit["agent_c_overlap"]
pd.DataFrame([
    {"partition": "Agent-C training", "evaluation_questions": overlap["evaluation_in_agent_c_training"]},
    {"partition": "Agent-C validation", "evaluation_questions": overlap["evaluation_in_agent_c_validation"]},
    {"partition": "Excluded from Agent-C", "evaluation_questions": overlap["evaluation_excluded_from_agent_c"]},
])


,partition,evaluation_questions
0,Agent-C training,202
1,Agent-C validation,38
2,Excluded from Agent-C,60


### GPQA currently has no usable difficulty variation

A rank correlation requires variation in the target. A constant difficulty value makes Spearman correlation undefined.


In [6]:
dataset_profiles = pd.DataFrame(audit["dataset_profiles"])
dataset_profiles[[
    "file", "rows", "unique_question_ids", "difficulty_column",
    "difficulty_complete_rows", "difficulty_unique_values",
    "difficulty_min", "difficulty_max", "difficulty_std"
]]


,file,rows,unique_question_ids,difficulty_column,difficulty_complete_rows,difficulty_unique_values,difficulty_min,difficulty_max,difficulty_std
0,data_preparation/mmlu_pro_hard_subset.json,3016,3016,difficulty_score,3016,1267,0.4117,0.9867,0.106068
1,data_preparation/gpqa_diamond.json,198,198,difficulty_score,198,1,1.0000,1.0000,0.000000


### A deterministic split repair is ready, but it is not yet wired into training

The manifest is stratified by category and difficulty quintile. It assigns every one of the 3,016 MMLU-Pro hard questions exactly once to train, validation, or test. Generating a manifest does not repair the submitted result by itself: Agent-C must be retrained and all co-evolution selection must be restricted to the training partition.


In [7]:
from create_split_manifest import build_manifest, write_manifest

split_manifest = build_manifest()
split_summary = write_manifest()
assert split_manifest["question_id"].nunique() == len(split_manifest)
assert split_manifest["split"].value_counts().to_dict() == {
    "train": 2416, "validation": 300, "test": 300
}
pd.DataFrame([
    {
        "split": split,
        "rows": split_summary["counts"][split],
        "difficulty_mean": details["mean"],
        "difficulty_std": details["std"],
    }
    for split, details in split_summary["difficulty_range_by_split"].items()
])


,split,rows,difficulty_mean,difficulty_std
0,test,300,0.621809,0.107881
1,train,2416,0.623195,0.105762
2,validation,300,0.623793,0.106693


### Canonical rescoring supports both saved schemas

The canonical scorer handles heterogeneous `per_agent` rows and homogeneous `per_role` rows. It reproduces the stored revised heterogeneous score to within rounding tolerance and evaluates both the submitted score and two no-correctness variants. The target in this section remains the criticized surface heuristic, so these results test implementation and sensitivity rather than validate intrinsic difficulty.


In [8]:
from local_saved_output_analysis import main as run_local_analysis

run_local_analysis()
component_ablation = pd.read_csv(
    ARTIFACT_DIR / "local_rescoring" / "component_ablation.csv"
)
key_scores = [
    "D_full",
    "D_no_correctness_paper",
    "D_no_correctness_renormalized",
]
key_files = [
    "experiments/experiment_0/results/exp0_heterogeneous.jsonl",
    "experiments/experiment_0/results/exp0_heterogeneous_mistral_d_n300.jsonl",
]
component_ablation.loc[
    component_ablation["file"].isin(key_files)
    & component_ablation["score"].isin(key_scores),
    [
        "file", "score", "n", "spearman_rho",
        "bootstrap_ci_lower", "bootstrap_ci_upper", "permutation_p",
    ],
]


                                                                    file                         score   n  spearman_rho  bootstrap_ci_lower  bootstrap_ci_upper  permutation_p
               experiments/experiment_0/results/exp0_heterogeneous.jsonl                        D_full  50      0.153920           -0.134450            0.400872       0.284472
               experiments/experiment_0/results/exp0_heterogeneous.jsonl        D_no_correctness_paper  50      0.206973           -0.072639            0.456873       0.143386
               experiments/experiment_0/results/exp0_heterogeneous.jsonl D_no_correctness_renormalized  50      0.209664           -0.077852            0.455331       0.137386
     experiments/experiment_0/results/exp0_heterogeneous_mistral_d.jsonl                        D_full 300      0.002896           -0.102791            0.120475       0.960404
     experiments/experiment_0/results/exp0_heterogeneous_mistral_d.jsonl        D_no_correctness_paper 300     -0.018338

,file,score,n,spearman_rho,bootstrap_ci_lower,bootstrap_ci_upper,permutation_p
0,experiments/experiment_0/results/exp0_heteroge...,D_full,50,0.153920,-0.134450,0.400872,0.284472
1,experiments/experiment_0/results/exp0_heteroge...,D_no_correctness_paper,50,0.206973,-0.072639,0.456873,0.143386
2,experiments/experiment_0/results/exp0_heteroge...,D_no_correctness_renormalized,50,0.209664,-0.077852,0.455331,0.137386
14,experiments/experiment_0/results/exp0_heteroge...,D_full,300,0.002896,-0.102791,0.120475,0.960404
15,experiments/experiment_0/results/exp0_heteroge...,D_no_correctness_paper,300,-0.018338,-0.127397,0.102830,0.740826
16,experiments/experiment_0/results/exp0_heteroge...,D_no_correctness_renormalized,300,-0.020924,-0.131121,0.098506,0.708929


### Surface-proxy triage is exploratory only

Top-20% precision and enrichment are computed to validate the evaluation code. These values must not be presented as rebuttal evidence until the positive class is defined by independent pass-rate, IRT, or human difficulty.


In [9]:
surface_triage = pd.read_csv(
    ARTIFACT_DIR / "local_rescoring" / "surface_proxy_triage.csv"
)
surface_triage.loc[
    (surface_triage["file"] == "experiments/experiment_0/results/exp0_heterogeneous_mistral_d_n300.jsonl")
    & surface_triage["score"].isin(key_scores),
    ["score", "n", "precision", "recall", "enrichment_over_random"],
]


,score,n,precision,recall,enrichment_over_random
14,D_full,300,0.266667,0.266667,1.333333
15,D_no_correctness_paper,300,0.250000,0.250000,1.250000
16,D_no_correctness_renormalized,300,0.233333,0.233333,1.166667


## Takeaways

1. **Safe now:** static-output rescoring, label-free score ablations, descriptive agent-error analysis, and code-level split/label auditing.
2. **Not safe yet:** citing Table 3 as reproduced, treating duplicated files as independent runs, claiming held-out Agent-C evaluation, or reporting cross-benchmark difficulty correlations from constant targets.
3. **Split repair ready:** a tested 2,416/300/300 manifest exists, but it must still be enforced in Agent-C training and the co-evolution pipeline.
4. **Required next:** recover the original iteration-level artifacts and adapters, rerun on the strict split, and construct a non-constant independently grounded difficulty target.
5. The generated CSV and JSON files in `rebuttal/artifacts/` are the compact machine-readable audit record.
